In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import glob
import os
import optuna
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

base_path = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'

# --- (特徴量作成とデータ読み込み部分は前回と全く同じなので省略せずに実行してください) ---
def create_features(df):
    df = df.sort_values('MD').reset_index(drop=True)
    df['GR_diff'] = df['GR'].diff().fillna(0)
    df['Z_diff'] = df['Z'].diff().fillna(0)
    df['GR_roll_mean_10'] = df['GR'].rolling(window=10, min_periods=1).mean()
    df['GR_roll_std_10'] = df['GR'].rolling(window=10, min_periods=1).std().fillna(0)
    df['GR_roll_mean_50'] = df['GR'].rolling(window=50, min_periods=1).mean()
    return df

print("データを準備しています...")
train_files = glob.glob(f'{base_path}/train/*__horizontal_well.csv')
train_list = [create_features(pd.read_csv(f).assign(well_id=os.path.basename(f).split('__')[0])) for f in train_files]
train_df = pd.concat(train_list, ignore_index=True)

test_files = glob.glob(f'{base_path}/test/*__horizontal_well.csv')
test_list = []
for f in test_files:
    df = pd.read_csv(f)
    well_id = os.path.basename(f).split('__')[0]
    df['well_id'] = well_id
    df['id'] = well_id + '_' + df.index.astype(str)
    test_list.append(create_features(df))
test_df = pd.concat(test_list, ignore_index=True)

features = ['MD', 'X', 'Y', 'Z', 'GR', 'GR_diff', 'Z_diff', 'GR_roll_mean_10', 'GR_roll_std_10', 'GR_roll_mean_50']
target = 'TVT'
train_df[features] = train_df[features].fillna(0)
test_df[features] = test_df[features].fillna(0)
train_df = train_df.dropna(subset=[target])

# -------------------------------------------------------------
# ★ 新兵器：Optunaによるハイパーパラメータ自動探索
# -------------------------------------------------------------
print("Optunaで最適なパラメータを探索中...")

def objective(trial):
    # 探索するパラメータの範囲を設定
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'random_state': 42,
        'verbose': -1
    }
    
    gkf = GroupKFold(n_splits=5)
    oof_preds = np.zeros(len(train_df))
    
    for train_idx, val_idx in gkf.split(train_df, train_df[target], train_df['well_id']):
        X_tr, y_tr = train_df.iloc[train_idx][features], train_df.iloc[train_idx][target]
        X_va, y_va = train_df.iloc[val_idx][features], train_df.iloc[val_idx][target]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        oof_preds[val_idx] = model.predict(X_va)
        
    # 誤差（RMSE）を計算して返す
    rmse = np.sqrt(mean_squared_error(train_df[target], oof_preds))
    return rmse

# 探索の実行（今回はテストとして10回だけ探索します）
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)

print("\n=== 探索完了 ===")
print("見つかった最強のパラメータ:", study.best_params)
print("その時のスコア(RMSE):", study.best_value)

# -------------------------------------------------------------
# ★ 見つけた最強のパラメータで本番の学習と予測
# -------------------------------------------------------------
print("\n最強パラメータを使って最終モデルを学習します...")
best_params = study.best_params
best_params['random_state'] = 42
best_params['verbose'] = -1

gkf = GroupKFold(n_splits=5)
models = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, train_df[target], train_df['well_id'])):
    X_tr, y_tr = train_df.iloc[train_idx][features], train_df.iloc[train_idx][target]
    X_va, y_va = train_df.iloc[val_idx][features], train_df.iloc[val_idx][target]
    
    model = lgb.LGBMRegressor(**best_params)
    model.fit(X_tr, y_tr)
    models.append(model)

print("提出ファイルを作成しています...")
preds = np.zeros(len(test_df))
for model in models:
    preds += model.predict(test_df[features]) / len(models)

test_df['predicted_tvt'] = preds
sub = pd.read_csv(f'{base_path}/sample_submission.csv')
sub = sub.drop(columns=['tvt']).merge(test_df[['id', 'predicted_tvt']], on='id', how='left')
sub = sub.rename(columns={'predicted_tvt': 'tvt'})
sub['tvt'] = sub['tvt'].fillna(0.0)

sub[['id', 'tvt']].to_csv('submission.csv', index=False)
print("完了しました！submission.csv が作成されました。")